In [1]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType


StatementMeta(, 0b9b8140-4a22-4475-8523-1c9b1fcfe1ff, 3, Finished, Available, Finished, False)

In [2]:
from datetime import date , timedelta
start_date = date.today() - timedelta(7)

StatementMeta(, 0b9b8140-4a22-4475-8523-1c9b1fcfe1ff, 4, Finished, Available, Finished, False)

In [4]:
# df now is a Spark DataFrame containing JSON data
df = spark.read.option("multiline" , "true").json(f"Files/{start_date}_earthquake_data.json")
display(df)

StatementMeta(, 0b9b8140-4a22-4475-8523-1c9b1fcfe1ff, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e006cd02-472c-4d56-b9bf-f881710b8f0d)

In [7]:
# Reshape earthquake data by extracting and renaming key attributes for further analysis.
df = \
df.\
    select(
        'id',
        col('geometry.coordinates').getItem(0).alias('longitude'),
        col('geometry.coordinates').getItem(1).alias('latitude'),
        col('geometry.coordinates').getItem(2).alias('elevation'),
        col('properties.title').alias('title'),
        col('properties.place').alias('place_description'),
        col('properties.sig').alias('sig'),
        col('properties.mag').alias('mag'),
        col('properties.magType').alias('magType'),
        col('properties.time').alias('time'),
        col('properties.updated').alias('updated')
        )


StatementMeta(, 0b9b8140-4a22-4475-8523-1c9b1fcfe1ff, 9, Finished, Available, Finished, False)

In [9]:
# Convert 'time' and 'updated' columns from milliseconds to timestamp format for clearer datetime representation.
df = df.\
withColumn('time', col('time')/1000).\
withColumn('updated', col('updated')/1000).\
withColumn('time', col('time').cast(TimestampType())).\
withColumn('updated', col('updated').cast(TimestampType()))

StatementMeta(, 0b9b8140-4a22-4475-8523-1c9b1fcfe1ff, 11, Finished, Available, Finished, False)

In [11]:
# appending the data to the gold table
df.write.mode('append').saveAsTable('earthquake_events_silver')

StatementMeta(, 0b9b8140-4a22-4475-8523-1c9b1fcfe1ff, 13, Finished, Available, Finished, False)